# Dataset Profiling EDA

This notebook programmatically scans the dataset directories, extracts audio metadata (like sample rate, duration, channels), and plots the distributions. Use the findings here to populate the markdown documentation.

In [ ]:
import os
import glob
import wave
import contextlib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa

# Set base paths based on the local environment or Google Drive structure
DATA_ROOT = "../data/raw/" # Or point to your Google Drive path if running in Colab

sns.set_theme(style="whitegrid")

## Utility Functions

In [ ]:
def extract_audio_metadata_wav(file_path):
    """Extract metadata from a WAV file without fully loading the audio array."""
    try:
        with contextlib.closing(wave.open(file_path, 'rb')) as f:
            frames = f.getnframes()
            rate = f.getframerate()
            duration = frames / float(rate)
            channels = f.getnchannels()
            sampwidth = f.getsampwidth() * 8 # bit depth
            return {
                "file": file_path,
                "duration": duration,
                "sample_rate": rate,
                "channels": channels,
                "bit_depth": sampwidth,
                "error": False
            }
    except Exception as e:
        return {
            "file": file_path,
            "duration": None,
            "sample_rate": None,
            "channels": None,
            "bit_depth": None,
            "error": str(e)
        }

def analyze_dataset_directory(dataset_dir, ext="*.wav"):
    """Crawls a directory recursively for files and extracts their metadata."""
    files = glob.glob(os.path.join(dataset_dir, f"**/{ext}"), recursive=True)
    print(f"Found {len(files)} files matching {ext} in {dataset_dir}")
    
    metadata = []
    for f in files:
        if ext == "*.wav":
            metadata.append(extract_audio_metadata_wav(f))
        # You can add elif blocks here for other formats like .avi if you plan to extract audio first
        
    return pd.DataFrame(metadata)

def plot_metadata_distributions(df, dataset_name):
    """Plots duration, sample rates, and channel types distributions."""
    if df.empty or df['error'].all():
        print(f"No valid data to plot for {dataset_name}")
        return
        
    valid_df = df[~df['error'].astype(bool)].copy()
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"{dataset_name} Audio Profiling", fontsize=16)
    
    # Durations
    sns.histplot(valid_df['duration'], bins=30, ax=axes[0], kde=True)
    axes[0].set_title("Duration Distribution (s)")
    
    # Sample Rates
    sns.countplot(x='sample_rate', data=valid_df, ax=axes[1])
    axes[1].set_title("Sample Rates")
    
    # Channels
    valid_df['channel_type'] = valid_df['channels'].map({1: 'Mono', 2: 'Stereo'})
    sns.countplot(x='channel_type', data=valid_df, ax=axes[2])
    axes[2].set_title("Channels")
    
    plt.tight_layout()
    plt.show()
    
    # Summary strings for markdown
    print(f"\n=== {dataset_name} Markdown Summary ===")
    print(f"- **Total Duration:** {valid_df['duration'].sum()/3600:.2f} hours")
    print(f"- **Number of Files:** {len(valid_df)}")
    print(f"- **Average Clip Length:** {valid_df['duration'].mean():.2f}s")
,  
- **Sample Rates:** {valid_df['sample_rate'].value_counts().to_dict()}")
    print(f"- **Channels:** {valid_df['channel_type'].value_counts().to_dict()}")
    print("====================================\n")

## 1. DESED Analysis
*(Make sure data exists locally or mount Google Drive first)*

In [ ]:
desed_dir = os.path.join(DATA_ROOT, "DESED", "audio")
if os.path.exists(desed_dir):
    df_desed = analyze_dataset_directory(desed_dir, ext="*.wav")
    plot_metadata_distributions(df_desed, "DESED")
else:
    print(f"Directory not found: {desed_dir}")

## 2. CAUCAFall Analysis
Note: Since CAUCAFall is natively `.avi`, you might need to extract audio first using `ffmpeg` before running this, or modify `analyze_dataset_directory` to use MoviePy/FFmpeg.

In [ ]:
# Example implementation stub for analyzing AVI files
caucafall_dir = os.path.join(DATA_ROOT, "CAUCAFall")
if os.path.exists(caucafall_dir):
    df_cauca = analyze_dataset_directory(caucafall_dir, ext="*.avi")
    # Note: df_cauca will show errors unless you implement AVI audio metadata extraction
    # plot_metadata_distributions(df_cauca, "CAUCAFall")
else:
    print(f"Directory not found: {caucafall_dir}")

## 3. WaterLeakage Analysis

In [ ]:
water_dir = os.path.join(DATA_ROOT, "water_leakage_voice_data")
if os.path.exists(water_dir):
    df_water = analyze_dataset_directory(water_dir, ext="*.wav")
    plot_metadata_distributions(df_water, "Water Leakage")
else:
    print(f"Directory not found: {water_dir}")